# IKG Column Lineage Master Auto Refresh

Parses all SQL files from the IKG GitLab repository (`develop` branch) and extracts **column-level lineage** — tracing every target column back to its ultimate source tables and columns through CTEs, temp tables, JOINs, WHERE, HAVING, subqueries, CASE expressions, etc.

**Outputs:**
- Excel: `ikg_column_lineage_master_auto_refresh_<YYYYMMDD_HHMMSS>.xlsx`
- Greenplum table: `<schema>.ikg_column_lineage_master_auto_refresh` (DROP + RECREATE)

**Modes:**
- `use_local_sql=True` → parse SQL files on disk (no GitLab call needed)
- `use_local_sql=False` → fetch live from GitLab `develop` branch

## 1. Setup & Imports

In [ ]:
import os
import sys
import getpass
import pandas as pd
from pathlib import Path
from datetime import datetime
from IPython.display import display

# Add script directory to path if needed
SCRIPT_DIR = Path('.')  # Adjust if script is in a different folder
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

from ikg_column_lineage_master_auto_refresh import (
    extract_lineage_from_sql,
    _load_local_sql_files,
    _fetch_sql_files_from_gitlab,
    save_to_excel,
    save_to_greenplum,
    ensure_greenplum_schema,
    get_private_token,
    COLUMN_ORDER,
    OUTPUT_TABLE,
)

print('✅ Imports successful')
print('Columns:', COLUMN_ORDER)

## 2. Configuration

In [ ]:
# ─── CHOOSE MODE ──────────────────────────────────────────────────────────────
USE_LOCAL_SQL = True   # True = parse local files;  False = fetch from GitLab

# Local SQL path (used only when USE_LOCAL_SQL = True)
LOCAL_SQL_PATH = str(Path('.') / 'dags/ikg/scripts/sql')

# GitLab token (used only when USE_LOCAL_SQL = False)
# Leave as None to be prompted, or set directly:
GITLAB_TOKEN = None   # or os.environ.get('GENESIS_DDLC_IKG_GIT_SECRET')

# Greenplum password — leave None to be prompted at save time
GREENPLUM_PASSWORD = None   # or 'your_password'

# Set True to write to Greenplum
SAVE_TO_GREENPLUM = False

print(f'Mode: {"Local SQL" if USE_LOCAL_SQL else "GitLab"}')
print(f'Save to Greenplum: {SAVE_TO_GREENPLUM}')

## 3. Fetch SQL Files

In [ ]:
if USE_LOCAL_SQL:
    file_dict = _load_local_sql_files(LOCAL_SQL_PATH)
else:
    if GITLAB_TOKEN is None:
        GITLAB_TOKEN = getpass.getpass('Enter GitLab private token: ')
    file_dict = _fetch_sql_files_from_gitlab(GITLAB_TOKEN)

print(f'\n📂 Total SQL files loaded: {len(file_dict)}')

## 4. Parse Column Lineage

In [ ]:
all_rows = []
parse_errors = []

total = len(file_dict)
for i, (fpath, content) in enumerate(file_dict.items(), 1):
    if i % 100 == 0:
        print(f'  Parsing {i}/{total}...', end='\r')
    fname = Path(fpath).name
    process = Path(fpath).parent.name
    try:
        rows = extract_lineage_from_sql(content, fname, fpath, process)
        all_rows.extend(rows)
    except Exception as e:
        parse_errors.append({'file': fpath, 'error': str(e)})

print(f'\n✅ Parsed {total} files')
print(f'📊 Raw lineage records: {len(all_rows)}')
if parse_errors:
    print(f'⚠️  Parse errors: {len(parse_errors)}')
    display(pd.DataFrame(parse_errors))

## 5. Build DataFrame

In [ ]:
df = pd.DataFrame(all_rows, columns=COLUMN_ORDER)
df = df.drop_duplicates()
df['current_date_time'] = pd.to_datetime(df['current_date_time'])

print(f'📊 Final records (deduped): {len(df):,}')
print(f'🗂️  Unique target tables:   {df["target_table"].nunique()}')
print(f'🗂️  Unique source tables:   {df["source_table"].nunique()}')
print(f'📁 Files processed:         {df["file"].nunique()}')
print()
print('sql_process breakdown:')
display(df['sql_process'].value_counts().reset_index())
df.head(10)

## 6. Quick Analysis

In [ ]:
print('=== Top 20 Target Tables by Unique Column Count ===')
display(
    df[df['target_table'] != '']
    .groupby('target_table')['target_column']
    .nunique()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
    .rename(columns={'target_column': 'unique_target_columns'})
)

In [ ]:
# Lookup: find all lineage for a specific target table
LOOKUP_TABLE = 'fee_waiver_rma_ikg'   # ← change this

result = df[df['target_table'] == LOOKUP_TABLE]
print(f'Lineage for [{LOOKUP_TABLE}]: {len(result)} records')
display(result[['target_column','source_table','source_schema',
                'source_column','logic','sql_process']].head(50))

In [ ]:
# Lookup: find all tables that source from a given table
SOURCE_LOOKUP = 'base_feature_shhp_ikg'   # ← change this

result2 = df[df['source_table'] == SOURCE_LOOKUP]
print(f'Tables that read from [{SOURCE_LOOKUP}]: {result2["target_table"].nunique()}')
display(
    result2[['target_table','target_column','source_column','logic']]
    .drop_duplicates()
    .head(30)
)

## 7. Export to Excel

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = f'ikg_column_lineage_master_auto_refresh_{timestamp}.xlsx'

save_to_excel(df, output_file)
print(f'✅ Excel written: {output_file}')

## 8. Save to Greenplum (optional)

In [ ]:
if SAVE_TO_GREENPLUM:
    schema = ensure_greenplum_schema()
    if GREENPLUM_PASSWORD is None:
        GREENPLUM_PASSWORD = getpass.getpass('Greenplum password: ')
    success = save_to_greenplum(df, schema, GREENPLUM_PASSWORD)
    if success:
        print(f'✅ Saved to {schema}.{OUTPUT_TABLE}')
    else:
        print('❌ Greenplum save failed — check logs above.')
else:
    print('ℹ️  Greenplum save skipped (SAVE_TO_GREENPLUM = False)')

## 9. Column Lineage Schema Reference

| Column | Description |
|---|---|
| `file` | SQL filename with extension (e.g. `account_profile_curr_ikg.sql`) |
| `path` | Full relative path of the SQL file in the repo |
| `target_table` | Final output table for this script (matches filename stem) |
| `target_schema` | Schema of target_table (may be a Jinja param like `IKG_SCHEMA`) |
| `process` | Parent folder of the SQL file (DAG group) |
| `current_date_time` | Timestamp when lineage was extracted |
| `sub_target_table` | Every table created in this script including temp tables and CTEs |
| `sub_target_schema` | Schema of sub_target_table |
| `target_column` | Column being written into sub_target_table |
| `source_table` | Real source table (CTE aliases resolved to actual tables) |
| `source_schema` | Schema of source_table |
| `source_column` | Column from source_table feeding target_column |
| `logic` | Raw SQL expression for this mapping |
| `sql_process` | Clause type: `select`, `select-value`, `select*`, `join`, `where`, `having` |